# Prosody Similarity Visualization

Compares pitch (F0) and intensity curves between a model speaker and
multiple learner recordings using the Vocametrix prosody similarity API.

Produces overlaid pitch curves and a similarity score bar chart.

**Requirements:** `pip install requests python-dotenv matplotlib numpy`

**Setup:** Copy `.env.example` to `.env` at the repo root and add your `VOCAMETRIX_API_KEY`.

In [ ]:
import os
import glob
import requests
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.cm as cm
from dotenv import load_dotenv

load_dotenv('../.env')
API_KEY = os.environ['VOCAMETRIX_API_KEY']
BASE_URL = 'https://platform.vocametrix.com'
HEADERS = {'X-API-Key': API_KEY}
print('API key loaded:', API_KEY[:6] + '...')

## Configuration

Set `MODEL_FILE` to the reference speaker recording and `LEARNER_FILES` to
a list of learner recordings (or a glob pattern).

In [ ]:
MODEL_FILE = '../test_audio/sustained_vowel.wav'  # Reference/model recording
LEARNER_GLOB = '../test_audio/*.wav'              # All WAVs except the model

learner_files = [f for f in sorted(glob.glob(LEARNER_GLOB))
                 if os.path.abspath(f) != os.path.abspath(MODEL_FILE)]

print(f'Model:    {os.path.basename(MODEL_FILE)}')
print(f'Learners: {[os.path.basename(f) for f in learner_files]}')

## Upload files and compute similarity

In [ ]:
def assign_file_id(audio_path, label=''):
    print(f'  Uploading {label or os.path.basename(audio_path)}...')
    with open(audio_path, 'rb') as f:
        r = requests.post(f'{BASE_URL}/api/assignFileId', headers=HEADERS,
                          files={'audio': f}, data={'email': 'user@example.com'})
    r.raise_for_status()
    return r.json()['fileId']

def prosody_similarity(model_id, learner_path):
    learner_id = assign_file_id(learner_path)
    r = requests.get(f'{BASE_URL}/api/calculate-prosody-similarity', headers=HEADERS,
                     params={'svFileId': model_id, 'csFileId': learner_id})
    r.raise_for_status()
    return r.json()

print('Uploading model...')
model_id = assign_file_id(MODEL_FILE, 'model')

results = []
for lf in learner_files:
    name = os.path.basename(lf)
    print(f'Computing similarity for {name}...')
    try:
        result = prosody_similarity(model_id, lf)
        score = result.get('PROSODY_SIMILARITY_SCORE', result.get('similarity_score', None))
        print(f'  score={score}')
        results.append({'name': name, 'score': score, 'data': result})
    except Exception as e:
        print(f'  ERROR: {e}')
        results.append({'name': name, 'score': None, 'data': {}, 'error': str(e)})

print(f'\nDone. {len(results)} comparisons.')

## Pitch curve overlay

In [ ]:
colors = cm.tab10(np.linspace(0, 0.9, max(len(results), 1)))

fig, axes = plt.subplots(len(results), 1, figsize=(12, 4 * len(results)), squeeze=False)

model_pitch = None
for r in results:
    if r['data'].get('PITCH_CURVE_MODEL'):
        model_pitch = r['data']['PITCH_CURVE_MODEL']
        break

for i, (r, color) in enumerate(zip(results, colors)):
    ax = axes[i][0]
    learner_pitch = r['data'].get('PITCH_CURVE_LEARNER', [])

    if model_pitch:
        t_model = np.linspace(0, 1, len(model_pitch))
        ax.plot(t_model, model_pitch, color='black', linewidth=2.5,
                linestyle='--', label='Model', zorder=3)

    if learner_pitch:
        t_learner = np.linspace(0, 1, len(learner_pitch))
        ax.plot(t_learner, learner_pitch, color=color, linewidth=1.8,
                label=f"{r['name']} (score={r['score']})", zorder=2)
        ax.fill_between(t_learner, learner_pitch, alpha=0.12, color=color)
    else:
        ax.text(0.5, 0.5, 'No pitch curve data', transform=ax.transAxes,
                ha='center', va='center', fontsize=12, color='gray')

    score_str = f'{r["score"]:.3f}' if isinstance(r.get('score'), float) else str(r.get('score', 'N/A'))
    ax.set_title(f"{r['name']}  —  similarity = {score_str}", fontweight='bold')
    ax.set_xlabel('Normalized time')
    ax.set_ylabel('F0 (Hz)')
    ax.legend(loc='upper right', fontsize=9)
    ax.grid(True, alpha=0.3)

fig.suptitle('Prosody Similarity — Pitch Curve Comparison', fontsize=14, fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig('prosody_curves.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved prosody_curves.png')

## Similarity score bar chart

In [ ]:
valid = [(r['name'], r['score']) for r in results if isinstance(r.get('score'), (int, float))]

if valid:
    names, scores = zip(*valid)
    bar_colors = cm.RdYlGn(np.array(scores))  # red=low, green=high

    fig, ax = plt.subplots(figsize=(max(6, len(valid) * 1.5), 5))
    bars = ax.bar(range(len(names)), scores, color=bar_colors, edgecolor='gray', linewidth=0.5)
    ax.set_xticks(range(len(names)))
    ax.set_xticklabels(names, rotation=30, ha='right', fontsize=10)
    ax.set_ylabel('Prosody Similarity Score')
    ax.set_title('Similarity vs Model — Per Learner', fontweight='bold')
    ax.set_ylim(0, 1)
    ax.axhline(0.8, color='green', linestyle='--', alpha=0.6, label='0.8 — good match')
    ax.axhline(0.5, color='orange', linestyle='--', alpha=0.6, label='0.5 — poor match')
    ax.legend()
    ax.grid(True, axis='y', alpha=0.3)

    for bar, score in zip(bars, scores):
        ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.01,
                f'{score:.2f}', ha='center', va='bottom', fontsize=9)

    plt.tight_layout()
    plt.savefig('prosody_scores.png', dpi=150, bbox_inches='tight')
    plt.show()
    print('Saved prosody_scores.png')
else:
    print('No valid scores to plot.')